In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from pathlib import Path
import asyncio
from hedging_dataset_creator.hedging_labeller import hedging_labeller
from hedging_dataset_creator.label_transcript import label_transcript_sentences
from hedging_dataset_creator.sentence_tokenizer import sentence_tokenizer
from speech_parser.transcript_parser import parse_transcript_to_json
import pandas as pd

0      Thank you and welcome to AMD's fourth-quarter ...
1                                  This is Laura Graves.
2      I recently joined the AMD IR team as Corporate...
3      By now you should have had the opportunity to ...
4      If you have not reviewed these documents, they...
                             ...                        
252                                 Thank you very much.
253                                 Thank you, operator.
254    Thank you, everyone, for joining us on our cal...
255    We look forward to speaking with you throughou...
256                                           Thank you.
Name: sentence, Length: 257, dtype: object

In [24]:
COMPANY = 'NVDA'
transcript_dir = Path(f"data/Transcripts/{COMPANY}")
transcript_files = sorted(transcript_dir.glob("*.txt"))
COOLDOWN_SECONDS = 60

print(f"Found {len(transcript_files)} {COMPANY} transcripts")
for idx, transcript_file in enumerate(transcript_files):
    print(f"Processing {transcript_file.name}")
    await label_transcript_sentences(str(transcript_file))
    if idx < len(transcript_files) - 1:
        print(f"Cooldown: waiting {COOLDOWN_SECONDS} seconds before next transcript...")
        await asyncio.sleep(COOLDOWN_SECONDS)

print(f"Done processing {COMPANY} transcripts")


Found 19 NVDA transcripts
Processing 2016-Aug-11-NVDA.txt


Labelling hedging sentences: 100%|██████████| 27/27 [00:04<00:00,  6.73batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2016-Feb-17-NVDA.txt


Labelling hedging sentences: 100%|██████████| 28/28 [00:05<00:00,  5.56batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2016-May-12-NVDA.txt


...

Cooldown: waiting 60 seconds before next transcript...
Processing 2020-Feb-13-NVDA.txt


Labelling hedging sentences: 100%|██████████| 21/21 [00:03<00:00,  6.96batch/s]


Cooldown: waiting 60 seconds before next transcript...
Processing 2020-May-21-NVDA.txt


Labelling hedging sentences: 100%|██████████| 36/36 [00:21<00:00,  1.66batch/s]

Done processing NVDA transcripts


In [31]:
csv_dir = Path(f"data/hedging_dataset/{COMPANY}")
csv_files = sorted(csv_dir.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found in {csv_dir}")
else:
    issues_found = False

    for csv_path in csv_files:
        print(csv_path)
        df = pd.read_csv(csv_path)

        if "isHedge" not in df.columns:
            issues_found = True
            print(f"\n[Missing column] {csv_path.name} has no 'isHedge' column.")
            continue

        # Convert to numeric; non-numeric values become NaN
        numeric_col = pd.to_numeric(df["isHedge"], errors="coerce")

        # Valid values are exactly 0 or 1 (0.0 and 1.0 naturally pass)
        invalid_mask = numeric_col.isna() | ~numeric_col.isin([0, 1])

        if invalid_mask.any():
            issues_found = True
            bad_rows = df.loc[invalid_mask, ["isHedge"]].copy()
            bad_rows["row_index"] = bad_rows.index
            print(f"\n[Invalid values] {csv_path.name}")
            print(bad_rows[["row_index", "isHedge"]].to_string(index=False))

    if not issues_found:
        print("All files passed: every 'isHedge' value is numeric 0/1 (including 0.0/1.0).")

data\hedging_dataset\NVDA\2016-Aug-11-NVDA.csv
data\hedging_dataset\NVDA\2016-Feb-17-NVDA.csv
data\hedging_dataset\NVDA\2016-May-12-NVDA.csv
data\hedging_dataset\NVDA\2016-Nov-10-NVDA.csv
data\hedging_dataset\NVDA\2017-Aug-10-NVDA.csv
data\hedging_dataset\NVDA\2017-Feb-09-NVDA.csv
data\hedging_dataset\NVDA\2017-May-09-NVDA.csv
data\hedging_dataset\NVDA\2017-Nov-09-NVDA.csv
data\hedging_dataset\NVDA\2018-Aug-16-NVDA.csv
data\hedging_dataset\NVDA\2018-Feb-08-NVDA.csv
data\hedging_dataset\NVDA\2018-May-10-NVDA.csv
data\hedging_dataset\NVDA\2018-Nov-15-NVDA.csv
data\hedging_dataset\NVDA\2019-Aug-15-NVDA.csv
data\hedging_dataset\NVDA\2019-Feb-14-NVDA.csv
data\hedging_dataset\NVDA\2019-May-16-NVDA.csv
data\hedging_dataset\NVDA\2019-Nov-14-NVDA.csv
data\hedging_dataset\NVDA\2020-Aug-19-NVDA.csv
data\hedging_dataset\NVDA\2020-Feb-13-NVDA.csv
data\hedging_dataset\NVDA\2020-May-21-NVDA.csv
All files passed: every 'isHedge' value is numeric 0/1 (including 0.0/1.0).


In [28]:
await label_transcript_sentences("data/Transcripts/NVDA/2020-May-21-NVDA.txt")

Labelling hedging sentences: 100%|██████████| 36/36 [00:23<00:00,  1.51batch/s]


,sentence,isHedge
0,Thank you.,0.0
1,"Good afternoon, everyone, and welcome to NVIDI...",0.0
2,With me on the call today from NVIDIA are Jens...,0.0
3,I'd like to remind you that our call is being ...,0.0
4,The webcast will be available for replay until...,0.0
...,...,...
710,"Third, we are opening large new markets with A...",0.0
711,"And then finally, we have built up multiple en...",0.0
712,"RTX computer graphics, artificial intelligence...",0.0
713,I look forward to updating you on our progress...,1.0
